In [ ]:
from src.misc import *
from src.SPECTRUM import Spectrum
from src.FITSPECTRUM import FitSpectrum
from src.DP import DP
import matplotlib.pyplot as plt
import numpy as np

In [2]:
spectra_data    = fits.open('/Users/hyp0515/data/0715_Spring_BGS_ALL_trimmed.fits')
cigale_data     = fits.open('/Users/hyp0515/data/IronPhysProp_v1.2_extracted.fits')
fastspecfit     = fits.open('/Users/hyp0515/data/0715_Spring_half_BGS_BRIGHT_catalog_fastspecfit.fits')

# Select parent sources who have at least one significant emission line (>5 S/N ratio)

In [3]:
FIT = FitSpectrum()
DP = DP()

ALL_SPECTRA = Spectrum(spectra_data, cigale_data, fastspecfit, load_targetID=None, subtype_filter='QSO')
print('Total number of spectra:', len(ALL_SPECTRA.targetID))

Data loading and processing took 39.68 seconds.
Total number of spectra: 99812


In [4]:
ALL_SPECTRA = FIT.label_emission_lines(ALL_SPECTRA, 5)
ALL_SPECTRA = FIT.significant_emission_filter(ALL_SPECTRA)
print('Total number of spectra:', len(ALL_SPECTRA.targetID))

/opt/anaconda3/envs/master_project/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/anaconda3/envs/master_project/lib/python3.13/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)


Total number of spectra: 66700


In [5]:
ALL_SPECTRA = ALL_SPECTRA.shrink_dataset(10)
print('Number of spectra after shrinking:', len(ALL_SPECTRA.targetID))

Number of spectra after shrinking: 6670


In [6]:
ALL_SPECTRA.df.head()

,TARGETID,RA,DEC,Z,SPECTYPE,LOGM,LOGSFR,OII,Hbeta,OIII,Halpha,NII,SII
0,39627739380056197,177.003822,-1.903397,0.224113,GALAXY,10.250155,-0.348563,True,False,False,False,False,False
1,39627739380057673,177.064329,-1.961656,0.168938,GALAXY,10.691324,-7.262499,True,False,False,False,False,False
2,39627739380059721,177.154096,-1.887425,0.104218,GALAXY,9.726343,-7.803186,True,False,False,False,False,False
3,39627739384251743,177.302871,-1.961315,0.123653,GALAXY,9.779942,0.461850,True,False,False,False,False,False
4,39627739388447756,177.619255,-1.974456,0.060054,GALAXY,9.651594,-3.581857,True,False,False,False,False,False


In [7]:
dp_parent, model_1comp, left_2comp, right_2comp = DP.fit_all(data_class=ALL_SPECTRA, n_jobs=10)

100%|██████████| 6670/6670 [00:52<00:00, 126.44it/s]


In [8]:
dp_parent[dp_parent['p_value']<0.05].head(30)

,TARGETID,RA,DEC,Z,LOGM,LOGSFR,dv_r,dv_l,sigma_r,sigma_l,...,OII3726_rank,OII3729_rank,Hbeta_rank,OIII4959_rank,OIII5007_rank,NII6548_rank,Halpha_rank,NII6583_rank,SII6716_rank,SII6731_rank
5,39627739392642816,177.899139,-1.878202,0.080528,9.211570,-0.308620,8.205591e+00,-8.171574,0.010000,33.379452,...,9,8,3,7,2,5,0,1,4,6
7,39627739401027885,178.263794,-1.890516,0.281338,10.514447,0.476694,5.875818e+01,-53.715061,98.254791,34.870026,...,7,5,3,9,8,4,0,1,6,2
8,39627739401030173,178.364517,-1.937933,0.165129,10.279449,0.333175,6.445075e+01,-55.683086,28.799391,37.895473,...,9,8,3,7,4,6,0,1,2,5
9,39627739405223043,178.544022,-1.945659,0.375717,10.712567,0.870017,8.666905e+01,-57.181076,75.833290,126.029007,...,6,2,8,9,3,7,0,1,4,5
10,39627739405227788,178.747360,-1.990688,0.176721,9.596325,0.350646,1.240877e-16,-2.226653,60.150059,18.940838,...,8,5,3,7,1,9,0,2,4,6
13,39627739417810260,179.468552,-1.990053,0.130946,10.196274,-1.838907,3.574602e+01,-35.958454,0.010000,52.803745,...,7,6,9,8,3,4,0,2,5,1
16,39627739430388674,180.037094,-1.939886,0.285738,10.732430,0.778315,3.554104e+01,-130.357071,63.710556,27.219936,...,8,6,2,9,5,4,0,1,3,7
23,39627739451362988,181.388260,-1.983970,0.080750,10.027017,0.412109,1.536307e+01,-72.411186,27.557989,41.722652,...,6,5,8,9,7,4,0,1,2,3
26,39627739459751770,181.897995,-1.975240,0.174288,10.261477,0.373320,5.444502e-03,-178.985565,63.677132,148.399170,...,7,6,3,9,8,2,0,1,4,5
29,39627739468138950,182.351044,-1.938127,0.158024,10.552460,1.079776,3.217157e+01,-122.639946,71.729027,35.939976,...,7,6,2,9,8,5,0,1,3,4


In [9]:
print(len(dp_parent[dp_parent['p_value']<0.05]))

3044


In [10]:
filenames = {
    'all'       : './catalogs/0315_all_test.fits',
    'dps'       : './catalogs/0315_dps_test.fits',
    'cs'        : './catalogs/0315_cs_test.fits',
    'nbcs'      : './catalogs/0315_nbcs_test.fits',
    'cs-nbcs'   : './catalogs/0315_cs-nbcs_test.fits'
}

In [11]:
# # dp_parent, model_1comp, left_2comp, right_2comp = DP.extract_fits_data(filenames['all'])
# # dp_parent = DP.bpt_classification(dp_parent, sigmas=1/np.sqrt(np.abs(ALL_SPECTRA.ivar), model_1comp=model_1comp, left_2comp=left_2comp, right_2comp=right_2comp, two_comp=True)
dp_sample, model_1comp, left_2comp, right_2comp = DP.select_dp_sample(dp_parent, model_1comp, left_2comp, right_2comp)

In [12]:
# print(len(dp_sample))

In [13]:
# dp_sample.head()

In [14]:
# line_cols = ['OII3726', 'OII3729',
#         'Hbeta',
#         'OIII4959', 'OIII5007',
#         'NII6548', 'Halpha', 'NII6583', 
#         'SII6716', 'SII6731']
# dp_cols = [f'{col}_dp' for col in line_cols]
# dp_rank_cols = [f'{col}_rank' for col in line_cols]
# flux_1comp_cols = [f'{col}_1comp' for col in line_cols]
# flux_2compL_cols = [f'{col}_2compL' for col in line_cols]
# flux_2compR_cols = [f'{col}_2compR' for col in line_cols]

In [15]:
# dp_parent.drop(columns=dp_rank_cols, inplace=True)
# dp_sample.drop(columns=dp_rank_cols, inplace=True)

In [16]:
# dp_parent.head(20)

In [17]:

DP.get_catalog(df=dp_parent, fname=filenames['all'], model_1comp=model_1comp, left_2comp=left_2comp, right_2comp=right_2comp)
DP.get_catalog(df=dp_sample, fname=filenames['dps'], model_1comp=model_1comp, left_2comp=left_2comp, right_2comp=right_2comp)

In [18]:
# dp_parent, model_1comp, left_2comp, right_2comp = DP.extract_fits_data(filenames['all'])
# dps_df, _, _, _ = DP.extract_fits_data(filenames['dps'])
# cs_df, nbcs_df, cs_nbcs_df = DP.select_nbcs(dp_parent=dp_parent, dp_sample=dps_df)

# cs_df.drop(columns=dp_cols+dp_rank_cols+flux_2compL_cols+flux_2compR_cols, inplace=True)
# nbcs_df.drop(columns=dp_cols+dp_rank_cols+flux_1comp_cols+flux_2compL_cols+flux_2compR_cols, inplace=True)
# cs_nbcs_df.drop(columns=dp_cols+dp_rank_cols+flux_1comp_cols+flux_2compL_cols+flux_2compR_cols, inplace=True)

# DP.get_catalog(cs_df, model_1comp=model_1comp[cs_df.index], left_2comp=left_2comp[cs_df.index], right_2comp=right_2comp[cs_df.index], fname=filenames['cs'])
# DP.get_catalog(nbcs_df, model_1comp=model_1comp[nbcs_df.index], left_2comp=left_2comp[nbcs_df.index], right_2comp=right_2comp[nbcs_df.index], fname=filenames['nbcs'])
# DP.get_catalog(cs_nbcs_df, model_1comp=model_1comp[cs_nbcs_df.index], left_2comp=left_2comp[cs_nbcs_df.index], right_2comp=right_2comp[cs_nbcs_df.index], fname=filenames['cs-nbcs'])